# Benchmark Models



## 1. Evaluation Framework

The modelling sample begins in 2006Q1. An initial training window from 2006Q1 to 2016Q4 is used, with forecasts generated sequentially from 2017Q1 onward. After each forecast, the training window expands by one quarter.

In [1]:
import pandas as pd
import statsmodels.api as sm

from statsmodels.stats.diagnostic import acorr_breusch_godfrey
from statsmodels.stats.stattools import durbin_watson
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

In [2]:
dataset_m1 = pd.read_csv("../data/processed/dataset_m1.csv")
dataset_m2 = pd.read_csv("../data/processed/dataset_m2.csv")
dataset_m3 = pd.read_csv("../data/processed/dataset_m3.csv")

In [3]:
for df in [dataset_m1, dataset_m2, dataset_m3]:
    df["quarter"] = pd.PeriodIndex(df["quarter"], freq="Q")

In [4]:
print("M1:", dataset_m1.shape)
print("M2:", dataset_m2.shape)
print("M3:", dataset_m3.shape)

print("\nSample:", dataset_m1["quarter"].min(), "to", dataset_m1["quarter"].max())

M1: (81, 4)
M2: (81, 8)
M3: (81, 20)

Sample: 2006Q1 to 2026Q1


In [5]:
print(
    "Same quarters:",
    dataset_m1["quarter"].equals(dataset_m2["quarter"])
    and dataset_m1["quarter"].equals(dataset_m3["quarter"])
)

Same quarters: True


In [6]:
evaluation_start = pd.Period("2017Q1", freq="Q")

In [7]:
initial_train = dataset_m3[
    dataset_m3["quarter"] < evaluation_start
]

evaluation_sample = dataset_m3[
    dataset_m3["quarter"] >= evaluation_start
]

print("Initial training observations:", len(initial_train))
print("Evaluation observations:", len(evaluation_sample))

print(
    "Training period:",
    initial_train["quarter"].min(),
    "to",
    initial_train["quarter"].max()
)

print(
    "Evaluation period:",
    evaluation_sample["quarter"].min(),
    "to",
    evaluation_sample["quarter"].max()
)

Initial training observations: 44
Evaluation observations: 37
Training period: 2006Q1 to 2016Q4
Evaluation period: 2017Q1 to 2026Q1


## 2. Naive Benchmark

The naive benchmark assumes that quarterly GDP growth in the current quarter is equal to GDP growth observed in the previous quarter:

$$
\hat{y}_t = y_{t-1}
$$

This provides a simple persistence benchmark against which the more sophisticated models can be evaluated.

In [8]:
for i in range(3):
    forecast_quarter = evaluation_sample.iloc[i]["quarter"]

    train = dataset_m3[
        dataset_m3["quarter"] < forecast_quarter
    ]

    naive_forecast = train["gdp_qoq"].iloc[-1]

    actual = evaluation_sample.iloc[i]["gdp_qoq"]

    print(
        f"{forecast_quarter}: "
        f"forecast = {naive_forecast}, "
        f"actual = {actual}"
    )

2017Q1: forecast = 0.6, actual = 1.0
2017Q2: forecast = 1.0, actual = 0.9
2017Q3: forecast = 0.9, actual = 0.7


In [9]:
naive_results = []

for i in range(len(evaluation_sample)):
    forecast_quarter = evaluation_sample.iloc[i]["quarter"]

    train = dataset_m3[
        dataset_m3["quarter"] < forecast_quarter
    ]

    naive_forecast = train["gdp_qoq"].iloc[-1]
    actual = evaluation_sample.iloc[i]["gdp_qoq"]

    naive_results.append({
        "quarter": forecast_quarter,
        "model": "Naive",
        "info_set": None,
        "actual": actual,
        "prediction": naive_forecast
    })

In [10]:
naive_results = pd.DataFrame(naive_results)

print("Number of forecasts:", len(naive_results))
display(naive_results.head())
display(naive_results.tail())

Number of forecasts: 37


,quarter,model,info_set,actual,prediction
0,2017Q1,Naive,None,1.0,0.6
1,2017Q2,Naive,None,0.9,1.0
2,2017Q3,Naive,None,0.7,0.9
3,2017Q4,Naive,None,0.8,0.7
4,2018Q1,Naive,None,0.1,0.8


,quarter,model,info_set,actual,prediction
32,2025Q1,Naive,None,0.6,0.4
33,2025Q2,Naive,None,0.2,0.6
34,2025Q3,Naive,None,0.1,0.2
35,2025Q4,Naive,None,0.1,0.1
36,2026Q1,Naive,None,0.6,0.1


## 3. Autoregressive Benchmark

An AR(1) model is used as a conventional time-series benchmark:

$$
GDP_t = \alpha + \beta GDP_{t-1} + \varepsilon_t
$$

The model is re-estimated at each step of the expanding window using only GDP observations available before the forecast quarter. Residual diagnostics on the initial training sample did not indicate significant first-order autocorrelation.

In [11]:
train = dataset_m3[
    dataset_m3["quarter"] < evaluation_start
][["quarter", "gdp_qoq"]].copy()

train["gdp_lag1"] = train["gdp_qoq"].shift(1)

display(train.head())

,quarter,gdp_qoq,gdp_lag1
0,2006Q1,0.4,NaN
1,2006Q2,0.3,0.4
2,2006Q3,0.2,0.3
3,2006Q4,0.4,0.2
4,2007Q1,1.1,0.4


In [12]:
train_ar = train.dropna().copy()

X_train = train_ar[["gdp_lag1"]]
y_train = train_ar["gdp_qoq"]

print("AR training observations", len(train_ar))
display(X_train.head())
display(y_train.head())

AR training observations 43


,gdp_lag1
1,0.4
2,0.3
3,0.2
4,0.4
5,1.1


1    0.3
2    0.2
3    0.4
4    1.1
5    0.9
Name: gdp_qoq, dtype: float64

In [13]:
X_ar_diag = sm.add_constant(X_train)

ar_diag = sm.OLS(y_train, X_ar_diag).fit()

bg_test = acorr_breusch_godfrey(
    ar_diag,
    nlags=1
)

print("Durbin-Watson:",
      round(durbin_watson(ar_diag.resid), 4))

print("Breusch-Godfrey p-value:",
      round(bg_test[1], 4))

Durbin-Watson: 1.8562
Breusch-Godfrey p-value: 0.5306


In [14]:
from sklearn.linear_model import LinearRegression
ar_model = LinearRegression()

ar_model.fit(X_train, y_train)

print("Intercept:", ar_model.intercept_)
print("AR(1) coefficient:", ar_model.coef_[0])

Intercept: 0.0888387096774193
AR(1) coefficient: 0.7257526881720432


In [15]:
ar_results = []

for i in range(len(evaluation_sample)):
    forecast_quarter = evaluation_sample.iloc[i]["quarter"]

    train = dataset_m3[
        dataset_m3["quarter"] < forecast_quarter][["quarter", "gdp_qoq"]].copy()

    train["gdp_lag1"] = train["gdp_qoq"].shift(1)
    train_ar = train.dropna().copy()

    X_train = train_ar[["gdp_lag1"]]
    y_train = train_ar["gdp_qoq"]

    ar_model = LinearRegression()
    ar_model.fit(X_train, y_train)

    last_gdp = train["gdp_qoq"].iloc[-1]

    X_forecast = pd.DataFrame({
        "gdp_lag1": [last_gdp]
    })

    ar_forecast = ar_model.predict(X_forecast)[0]
    actual = evaluation_sample.iloc[i]["gdp_qoq"]

    ar_results.append({
        "quarter": forecast_quarter,
        "model": "AR(1)",
        "info_set": None,
        "actual": actual,
        "prediction": ar_forecast
    })



In [16]:
ar_results = pd.DataFrame(ar_results)

print("Number of forecasts:", len(ar_results))
display(ar_results.head())
display(ar_results.tail())

Number of forecasts: 37


,quarter,model,info_set,actual,prediction
0,2017Q1,AR(1),None,1.0,0.524290
1,2017Q2,AR(1),None,0.9,0.829709
2,2017Q3,AR(1),None,0.7,0.759284
3,2017Q4,AR(1),None,0.8,0.610636
4,2018Q1,AR(1),None,0.1,0.689259


,quarter,model,info_set,actual,prediction
32,2025Q1,AR(1),None,0.6,0.349311
33,2025Q2,AR(1),None,0.2,0.290427
34,2025Q3,AR(1),None,0.1,0.413623
35,2025Q4,AR(1),None,0.1,0.440679
36,2026Q1,AR(1),None,0.6,0.436337


## 4. Ridge Regression

Ridge regression is used as a regularised linear benchmark for the M1, M2 and M3 information sets. Predictors are standardised within each expanding training window, and the model is re-estimated for every forecast quarter.


In [17]:
ridge_m1_results = []

for i in range(len(evaluation_sample)):
    forecast_quarter = evaluation_sample.iloc[i]["quarter"]

    ridge_train = dataset_m1[
        dataset_m1["quarter"] < forecast_quarter].copy()

    ridge_test = dataset_m1[
        dataset_m1["quarter"] == forecast_quarter].copy()

    feature_cols = [
        col for col in ridge_train.columns
        if col not in ["quarter", "gdp_qoq"]
        ]

    X_train = ridge_train[feature_cols]
    y_train = ridge_train["gdp_qoq"]

    X_test = ridge_test[feature_cols]

    ridge_pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("ridge", Ridge())
    ])

    param_grid = {
        "ridge__alpha": [0.01, 0.1, 1, 10, 100, 1000, 10000]
    }

    tscv = TimeSeriesSplit(n_splits=5)

    ridge_search = GridSearchCV(
        estimator=ridge_pipeline,
        param_grid=param_grid,
        cv=tscv,
        scoring="neg_mean_squared_error"
    )

    ridge_search.fit(X_train, y_train)

    ridge_forecast = ridge_search.predict(X_test)[0]
    actual = ridge_test["gdp_qoq"].iloc[0]

    ridge_m1_results.append({
        "quarter": forecast_quarter,
        "model": "Ridge",
        "info_set": "M1",
        "actual": actual,
        "prediction": ridge_forecast,
        "alpha": ridge_search.best_params_["ridge__alpha"]
    })

In [18]:
ridge_m1_results = pd.DataFrame(ridge_m1_results)

print("Number of forecasts:", len(ridge_m1_results))
display(ridge_m1_results.head())
display(ridge_m1_results.tail())

Number of forecasts: 37


,quarter,model,info_set,actual,prediction,alpha
0,2017Q1,Ridge,M1,1.0,0.445375,100
1,2017Q2,Ridge,M1,0.9,0.442845,100
2,2017Q3,Ridge,M1,0.7,0.405017,100
3,2017Q4,Ridge,M1,0.8,0.360609,1000
4,2018Q1,Ridge,M1,0.1,0.359631,10000


,quarter,model,info_set,actual,prediction,alpha
32,2025Q1,Ridge,M1,0.6,0.190476,100
33,2025Q2,Ridge,M1,0.2,0.193281,100
34,2025Q3,Ridge,M1,0.1,0.345599,1000
35,2025Q4,Ridge,M1,0.1,0.300900,100
36,2026Q1,Ridge,M1,0.6,0.321002,100


In [19]:
ridge_m1_results["alpha"].value_counts().sort_index()

alpha
100      29
1000      5
10000     3
Name: count, dtype: int64

In [20]:
m2_feature_cols = [
    col for col in dataset_m2.columns
    if col not in ["quarter", "gdp_qoq"]
]

print("Number of M2 features:", len(m2_feature_cols))
print("M2 features:")
print(m2_feature_cols)

Number of M2 features: 6
M2 features:
['bank_rate_m1', 'bank_rate_m2', 'consumer_confidence_m1', 'consumer_confidence_m2', 'cpi_inflation_m1', 'retail_sales_m1']


In [21]:
ridge_m2_results = []

for i in range(len(evaluation_sample)):
    forecast_quarter = evaluation_sample.iloc[i]["quarter"]

    ridge_train = dataset_m2[
        dataset_m2["quarter"] < forecast_quarter
    ].copy()

    ridge_test = dataset_m2[
        dataset_m2["quarter"] == forecast_quarter
    ].copy()

    feature_cols = [
        col for col in ridge_train.columns
        if col not in ["quarter", "gdp_qoq"]
    ]

    X_train = ridge_train[feature_cols]
    y_train = ridge_train["gdp_qoq"]

    X_test = ridge_test[feature_cols]

    ridge_pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("ridge", Ridge())
    ])

    param_grid = {
        "ridge__alpha": [0.01, 0.1, 1, 10, 100, 1000, 10000]
    }

    tscv = TimeSeriesSplit(n_splits=5)

    ridge_search = GridSearchCV(
        estimator=ridge_pipeline,
        param_grid=param_grid,
        cv=tscv,
        scoring="neg_mean_squared_error"
    )

    ridge_search.fit(X_train, y_train)

    ridge_forecast = ridge_search.predict(X_test)[0]
    actual = ridge_test["gdp_qoq"].iloc[0]

    ridge_m2_results.append({
        "quarter": forecast_quarter,
        "model": "Ridge",
        "info_set": "M2",
        "actual": actual,
        "prediction": ridge_forecast,
        "alpha": ridge_search.best_params_["ridge__alpha"]
    })

In [22]:
ridge_m2_results = pd.DataFrame(ridge_m2_results)

print("Number of forecasts:", len(ridge_m2_results))
display(ridge_m2_results.head())
display(ridge_m2_results.tail())

print("\nAlpha counts:")
print(ridge_m2_results["alpha"].value_counts().sort_index())

Number of forecasts: 37


,quarter,model,info_set,actual,prediction,alpha
0,2017Q1,Ridge,M2,1.0,0.348151,1000
1,2017Q2,Ridge,M2,0.9,0.482524,100
2,2017Q3,Ridge,M2,0.7,0.435083,100
3,2017Q4,Ridge,M2,0.8,0.360510,1000
4,2018Q1,Ridge,M2,0.1,0.359963,10000


,quarter,model,info_set,actual,prediction,alpha
32,2025Q1,Ridge,M2,0.6,0.218213,1000
33,2025Q2,Ridge,M2,0.2,0.312496,1000
34,2025Q3,Ridge,M2,0.1,0.455081,1000
35,2025Q4,Ridge,M2,0.1,0.276597,1000
36,2026Q1,Ridge,M2,0.6,0.483337,1000



Alpha counts:
alpha
100      17
1000     18
10000     2
Name: count, dtype: int64


In [23]:
m3_feature_cols = [
    col for col in dataset_m3.columns
    if col not in ["quarter", "gdp_qoq"]
]

print("Number of M3 features:", len(m3_feature_cols))
print("M3 features:")
print(m3_feature_cols)

Number of M3 features: 18
M3 features:
['bank_rate_m1', 'bank_rate_m2', 'bank_rate_m3', 'consumer_confidence_m1', 'consumer_confidence_m2', 'consumer_confidence_m3', 'cpi_inflation_m1', 'cpi_inflation_m2', 'retail_sales_m1', 'retail_sales_m2', 'oecd_cli_m1', 'gdp_monthly_m1', 'services_m1', 'industrial_production_m1', 'construction_m1', 'unemployment_rate_m1', 'vacancies_thousands_m1', 'wage_growth_m1']


In [24]:
ridge_m3_results = []

for i in range(len(evaluation_sample)):
    forecast_quarter = evaluation_sample.iloc[i]["quarter"]

    ridge_train = dataset_m3[
        dataset_m3["quarter"] < forecast_quarter
    ].copy()

    ridge_test = dataset_m3[
        dataset_m3["quarter"] == forecast_quarter
    ].copy()

    feature_cols = [
        col for col in ridge_train.columns
        if col not in ["quarter", "gdp_qoq"]
    ]

    X_train = ridge_train[feature_cols]
    y_train = ridge_train["gdp_qoq"]

    X_test = ridge_test[feature_cols]

    ridge_pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("ridge", Ridge())
    ])

    param_grid = {
        "ridge__alpha": [0.01, 0.1, 1, 10, 100, 1000, 10000]
    }

    tscv = TimeSeriesSplit(n_splits=5)

    ridge_search = GridSearchCV(
        estimator=ridge_pipeline,
        param_grid=param_grid,
        cv=tscv,
        scoring="neg_mean_squared_error"
    )

    ridge_search.fit(X_train, y_train)

    ridge_forecast = ridge_search.predict(X_test)[0]
    actual = ridge_test["gdp_qoq"].iloc[0]

    ridge_m3_results.append({
        "quarter": forecast_quarter,
        "model": "Ridge",
        "info_set": "M3",
        "actual": actual,
        "prediction": ridge_forecast,
        "alpha": ridge_search.best_params_["ridge__alpha"]
    })

In [25]:
ridge_m3_results = pd.DataFrame(ridge_m3_results)

print("Number of forecasts:", len(ridge_m3_results))
display(ridge_m3_results.head())
display(ridge_m3_results.tail())

print("\nAlpha counts:")
print(ridge_m3_results["alpha"].value_counts().sort_index())

Number of forecasts: 37


,quarter,model,info_set,actual,prediction,alpha
0,2017Q1,Ridge,M3,1.0,-0.016425,1.0
1,2017Q2,Ridge,M3,0.9,0.499023,100.0
2,2017Q3,Ridge,M3,0.7,0.379834,1000.0
3,2017Q4,Ridge,M3,0.8,0.372184,1000.0
4,2018Q1,Ridge,M3,0.1,0.364860,1000.0


,quarter,model,info_set,actual,prediction,alpha
32,2025Q1,Ridge,M3,0.6,1.035603,0.1
33,2025Q2,Ridge,M3,0.2,-0.003275,10.0
34,2025Q3,Ridge,M3,0.1,0.808303,0.1
35,2025Q4,Ridge,M3,0.1,-0.376184,10.0
36,2026Q1,Ridge,M3,0.6,0.614642,0.1



Alpha counts:
alpha
0.01         2
0.10        11
1.00         2
10.00        8
100.00       6
1000.00      7
10000.00     1
Name: count, dtype: int64


## 5. Save Forecast Results

Forecasts from the benchmark models are saved for use in the final model evaluation.

In [26]:
naive_results.to_csv(
    "../data/processed/naive_forecasts.csv",
    index=False
)

ar_results.to_csv(
    "../data/processed/ar_forecasts.csv",
    index=False
)

ridge_m1_results.to_csv(
    "../data/processed/ridge_m1_forecasts.csv",
    index=False
)

ridge_m2_results.to_csv(
    "../data/processed/ridge_m2_forecasts.csv",
    index=False
)

ridge_m3_results.to_csv(
    "../data/processed/ridge_m3_forecasts.csv",
    index=False
)

print("Benchmark forecasts saved.")

Benchmark forecasts saved.
